# 05 — Mutation-Aware ESM2 on TCGA-BRCA

Notebook 04 showed ESM2 added nothing over expression — because every patient got the **same reference protein sequence**, so the embeddings were just a linear re-mix of expression. Cross-validation confirmed the tiny "Combined" gain was a PCA-truncation artifact, not real information.

This notebook fixes that by giving ESM2 sequences that **differ between patients**: their somatic **mutations**. For each missense mutation we ask ESM2 how damaging it looks (a variant-effect score), build a per-patient feature matrix from those scores, and rerun Expression vs. Mutation-ESM2 vs. Combined with a cross-validated C-index.

**Heads-up:** this notebook is written to slot into your project but has **not** been run against your exact files. Column names and the mutation dataset URL vary between TCGA/Xena versions — a couple of cells auto-detect columns and print what they found, in the spirit of notebook 01. Adjust if needed.

Run from the `Breast_Cancer2` folder.

## The idea: ESM2 as a variant-effect predictor

This is what protein language models are actually built for. For a missense mutation (one amino acid swapped for another) at position *p*:

1. Take the wildtype protein sequence.
2. Mask position *p* and ask ESM2 for its probability distribution over the 20 amino acids at that spot (what does evolution "expect" here?).
3. Score the mutation as the **log-likelihood ratio** `logP(mutant) - logP(wildtype)`.

A large negative score = the model is surprised by the substitution = likely damaging. A score near zero = a tolerated swap. This is the "masked-marginal" method (Meier et al., 2021) and needs the masked-LM head of ESM2, so we load `EsmForMaskedLM` (notebook 04 used `EsmModel`, which has no LM head).

Crucially, two patients with identical expression but different mutations now get different features — the orthogonal signal expression cannot provide.

In [12]:
import re
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

import torch
from transformers import EsmTokenizer, EsmForMaskedLM

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt

RAW_DIR  = Path("data/raw")
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK.")

Imports OK.


## 1. Load somatic mutations (MAF)

We use the TCGA Pan-Cancer **MC3** public mutation set from UCSC Xena and keep only our BRCA patients by intersecting sample barcodes. MC3 is a per-variant table (one row per mutation) that includes the amino-acid change we need.

> If the URL 404s, open [xenabrowser.net/datapages](https://xenabrowser.net/datapages/), find a BRCA somatic-mutation dataset that has an `Amino_Acid_Change` column, and paste its URL here.

In [13]:
MUT_URL  = "https://pancanatlas.xenahubs.net/download/mc3.v0.2.8.PUBLIC.xena.gz"
MUT_FILE = RAW_DIR / "mc3_mutations.gz"

if not MUT_FILE.exists():
    print("Downloading MC3 mutations (large file, one-time)...")
    with requests.get(MUT_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(MUT_FILE, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print("  saved ->", MUT_FILE)

maf = pd.read_csv(MUT_FILE, sep="\t", compression="gzip", low_memory=False)
print("MAF shape:", maf.shape)
print("Columns:", list(maf.columns))
maf.head(3)

MAF shape: (2907335, 12)
Columns: ['sample', 'chr', 'start', 'end', 'reference', 'alt', 'gene', 'effect', 'Amino_Acid_Change', 'DNA_VAF', 'SIFT', 'PolyPhen']


,sample,chr,start,end,reference,alt,gene,effect,Amino_Acid_Change,DNA_VAF,SIFT,PolyPhen
0,TCGA-02-0003-01,10,123810032,123810032,C,T,TACC2,Missense_Mutation,p.T38M,0.88,NaN,benign(0.335)
1,TCGA-02-0003-01,10,133967449,133967449,C,T,JAKMIP3,Silent,p.D723D,0.79,NaN,NaN
2,TCGA-02-0003-01,11,124489539,124489539,G,A,PANX3,Missense_Mutation,p.R296Q,0.41,tolerated(0.37),benign(0.001)


## 2. Parse missense changes and restrict to a gene panel

Scoring every mutation in the genome is unnecessary and slow. We restrict to a panel of genes recurrently mutated in breast cancer, and to **missense** mutations only — the substitutions the LLR score is defined for (we skip silent, nonsense, frameshift and indel variants).

The amino-acid change looks like `p.H1047R`: wildtype **H**, position **1047**, mutant **R**.

In [14]:
# auto-detect the columns we need (names vary between Xena versions)
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"none of {candidates} found in {list(df.columns)}")

SAMPLE_COL = find_col(maf, ["sample", "Sample_ID", "sampleID", "Tumor_Sample_Barcode"])
GENE_COL   = find_col(maf, ["gene", "Hugo_Symbol", "genes"])
AA_COL     = find_col(maf, ["Amino_Acid_Change", "HGVSp_Short", "amino_acid_change", "AAChange"])
EFF_COL    = find_col(maf, ["effect", "Variant_Classification", "consequence"])
print("using columns ->", SAMPLE_COL, GENE_COL, AA_COL, EFF_COL)

PANEL = ["PIK3CA","TP53","CDH1","GATA3","MAP3K1","MAP2K4","PTEN","AKT1","ARID1A",
         "RB1","NF1","ERBB2","ERBB3","FOXA1","RUNX1","CBFB","KMT2C","NCOR1","TBX3",
         "SF3B1","CTCF","BRCA1","BRCA2","ATM","CHEK2","PALB2","KRAS"]

MISSENSE = {"Missense_Mutation", "missense_variant", "Missense", "missense"}
aa_re = re.compile(r"^p?\.?([ACDEFGHIKLMNPQRSTVWY])(\d+)([ACDEFGHIKLMNPQRSTVWY])$")

def parse_aa(change):
    if not isinstance(change, str):
        return None
    m = aa_re.match(change.strip())
    if not m:
        return None
    return m.group(1), int(m.group(2)), m.group(3)   # (wt, pos, mut)

# trim barcodes to sample level (TCGA-XX-XXXX-01)
def sample_level(bc):
    parts = str(bc).split("-")
    return "-".join(parts[:4]) if len(parts) >= 4 else str(bc)

sub = maf[maf[GENE_COL].isin(PANEL) & maf[EFF_COL].isin(MISSENSE)].copy()
sub["parsed"] = sub[AA_COL].map(parse_aa)
sub = sub[sub["parsed"].notna()]
sub["sample_id"] = sub[SAMPLE_COL].map(sample_level)
print(f"{len(sub)} panel missense mutations across {sub['sample_id'].nunique()} samples")
print(sub[GENE_COL].value_counts().head(10))

using columns -> sample gene Amino_Acid_Change effect
9777 panel missense mutations across 4693 samples
gene
TP53      2382
PIK3CA    1364
KMT2C      752
KRAS       682
BRCA2      457
PTEN       447
ATM        433
NF1        327
NCOR1      326
ERBB3      273
Name: count, dtype: int64


## 3. Wildtype protein sequences

We need the reference sequence for each panel gene to mutate and score. We reuse the UniProt cache from notebook 04 where possible and fetch any missing panel genes.

In [15]:
def fetch_uniprot_sequence(gene_name):
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {"query": f"gene_exact:{gene_name} AND organism_id:9606 AND reviewed:true",
              "fields": "sequence", "format": "json", "size": 1}
    try:
        r = requests.get(url, params=params, timeout=15)
        if r.status_code == 200 and r.json()["results"]:
            return r.json()["results"][0]["sequence"]["value"]
    except Exception:
        pass
    return None

seq_cache = PROC_DIR / "gene_sequences.pkl"
gene_sequences = {}
if seq_cache.exists():
    with open(seq_cache, "rb") as f:
        gene_sequences = pickle.load(f)

missing = [g for g in PANEL if g not in gene_sequences]
for g in tqdm(missing, desc="fetching panel sequences"):
    s = fetch_uniprot_sequence(g)
    if s:
        gene_sequences[g] = s
    time.sleep(0.15)

with open(seq_cache, "wb") as f:
    pickle.dump(gene_sequences, f)

have = [g for g in PANEL if g in gene_sequences]
print(f"sequences available for {len(have)}/{len(PANEL)} panel genes")

fetching panel sequences: 0it [00:00, ?it/s]

sequences available for 27/27 panel genes


## 4. Load ESM2 (masked-LM head) and define the variant-effect score

We load `EsmForMaskedLM` — it has the language-model head that gives per-position amino-acid probabilities. ESM tokens are `[CLS] r1 r2 ... rL [EOS]`, so residue at 1-based position *i* sits at token index *i* (CLS shifts everything by one). We check the wildtype letter matches the sequence before scoring — UniProt isoforms sometimes differ from the mutation call's transcript, and a mismatch means the position can't be trusted, so we skip it.

In [16]:
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
print("Loading", MODEL_NAME, "...")
tokenizer = EsmTokenizer.from_pretrained(MODEL_NAME)
mlm = EsmForMaskedLM.from_pretrained(MODEL_NAME)
mlm.eval()
MAX_LEN = 1022

def score_variant(sequence, wt, pos, mut):
    # returns logP(mut) - logP(wt) at position pos (1-based), or None if unscorable
    seq = sequence[:MAX_LEN]
    if pos > len(seq) or seq[pos - 1] != wt:   # truncated or isoform mismatch
        return None
    enc = tokenizer(seq, return_tensors="pt")
    ids = enc["input_ids"].clone()
    tok_idx = pos                              # +1 CLS offset
    ids[0, tok_idx] = tokenizer.mask_token_id
    with torch.no_grad():
        logits = mlm(ids).logits[0, tok_idx]
    logp = torch.log_softmax(logits, dim=-1)
    wt_id, mut_id = tokenizer.convert_tokens_to_ids(wt), tokenizer.convert_tokens_to_ids(mut)
    return float(logp[mut_id] - logp[wt_id])

print("Model ready.")

Loading facebook/esm2_t6_8M_UR50D ...


Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready.


## 5. Score every unique variant (cached)

Many patients share the same hotspot mutation (e.g. PIK3CA H1047R), so we score each **unique** (gene, change) once and reuse it — far fewer forward passes.

In [17]:
score_cache = PROC_DIR / "variant_scores.pkl"
if score_cache.exists():
    with open(score_cache, "rb") as f:
        variant_scores = pickle.load(f)
else:
    variant_scores = {}
    uniq = sub[[GENE_COL, "parsed"]].drop_duplicates()
    uniq = uniq[uniq[GENE_COL].isin(gene_sequences)]
    for gene, (wt, pos, mut) in tqdm(list(zip(uniq[GENE_COL], uniq["parsed"])),
                                     desc="scoring variants"):
        variant_scores[(gene, wt, pos, mut)] = score_variant(gene_sequences[gene], wt, pos, mut)
    with open(score_cache, "wb") as f:
        pickle.dump(variant_scores, f)

n_ok = sum(v is not None for v in variant_scores.values())
print(f"scored {n_ok}/{len(variant_scores)} unique variants "
      f"({len(variant_scores)-n_ok} skipped: truncated or isoform mismatch)")

scored 3113/5192 unique variants (2079 skipped: truncated or isoform mismatch)


## 6. Build the patient x gene damage matrix

One row per patient, one column per panel gene. The value is the variant-effect score of that patient's mutation in that gene (the **most damaging** one if there are several); 0 means no scored missense mutation there. Zero is a fair neutral default — it sits between damaging (negative) and tolerated (~0).

In [18]:
patients_mut = sorted(sub["sample_id"].unique())
X_mut_df = pd.DataFrame(0.0, index=patients_mut, columns=PANEL)

for sid, gene, (wt, pos, mut) in zip(sub["sample_id"], sub[GENE_COL], sub["parsed"]):
    s = variant_scores.get((gene, wt, pos, mut))
    if s is not None:
        # keep the most damaging (most negative) score per patient-gene
        X_mut_df.loc[sid, gene] = min(X_mut_df.loc[sid, gene], s)

nz = X_mut_df.values[X_mut_df.values != 0]
print("mutation feature matrix:", X_mut_df.shape)
print("mean non-zero score:", round(nz.mean(), 3) if nz.size else "n/a")

mutation feature matrix: (4693, 27)
mean non-zero score: -1.523


## 7. Align with expression and survival

We line up three feature blocks on the patients who have mutation data, expression, and survival: expression (top variable genes), mutation-ESM2 scores, and the two combined. Patients with no panel mutation keep an all-zero mutation row — itself information (few/no driver hits).

In [19]:
# expression: rebuild the top-variable-gene matrix the same way as nb04
expr = pd.read_csv(RAW_DIR / "expression.gz", sep="\t", index_col=0)
N_GENES = 500
top_genes = expr.var(axis=1).nlargest(N_GENES).index
expr_top = expr.loc[top_genes]
expr_top.columns = [sample_level(c) for c in expr_top.columns]
expr_top = expr_top.loc[:, ~expr_top.columns.duplicated()]   # drop duplicate aliquots

meta = pd.read_csv(PROC_DIR / "metadata.csv", index_col=0)
meta.index = [sample_level(i) for i in meta.index]
surv = meta[~meta.index.duplicated()].dropna(subset=["OS_time", "OS_event"])

# align on expression + survival ONLY — do NOT require a mutation
common = sorted(set(expr_top.columns) & set(surv.index))
print("patients with expression + survival:", len(common))

X_expr = expr_top[common].T.values
# reindex mutation matrix onto the full cohort; no panel missense -> all zeros (valid info)
X_mut  = X_mut_df.reindex(common).fillna(0.0).values
X_both = np.hstack([X_expr, X_mut])
T = surv.loc[common, "OS_time"].values.astype(float)
E = surv.loc[common, "OS_event"].values.astype(int)
print("deaths:", int(E.sum()), "/", len(E))
print("mutated patients in cohort:", int((X_mut != 0).any(axis=1).sum()))
print("shapes -> expr:", X_expr.shape, "| mut:", X_mut.shape, "| both:", X_both.shape)

patients with expression + survival: 801
deaths: 90 / 801
mutated patients in cohort: 190
shapes -> expr: (801, 500) | mut: (801, 27) | both: (801, 527)


## 8. Cross-validated C-index — does mutation-ESM2 add signal?

Same honest protocol as notebook 04's fix: PCA + scaler + Cox fit **inside** each training fold, score the held-out fold, report mean +/- std. If the mutation features carry independent prognostic signal, Combined should now beat Expression by more than the fold-to-fold noise — unlike the reference-sequence version, which didn't.

In [23]:
def cv_cindex(X, T, E, n_splits=5, n_components=20, penalizer=0.1, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = []
    for tr, te in skf.split(X, E):
        scaler = StandardScaler().fit(X[tr])
        pca = PCA(n_components=min(n_components, X.shape[1]), random_state=seed).fit(scaler.transform(X[tr]))
        def design(idx):
            Xp = pca.transform(scaler.transform(X[idx]))
            df = pd.DataFrame(Xp, columns=[f"PC{i}" for i in range(Xp.shape[1])])
            df["T"], df["E"] = T[idx], E[idx]
            return df
        cph = CoxPHFitter(penalizer=penalizer).fit(design(tr), duration_col="T", event_col="E")
        risk = cph.predict_partial_hazard(design(te)).values
        scores.append(concordance_index(T[te], -risk, E[te]))
    return np.array(scores)

print("Cross-validated C-index (5-fold, mean +/- std):\n")
for name, X in [("Expression only", X_expr), ("Mutation-ESM2 only", X_mut), ("Combined", X_both)]:
    s = cv_cindex(X, T, E)
    print(f"  {name:20s}: {s.mean():.3f} +/- {s.std():.3f}   (folds: {[f'{v:.3f}' for v in s]})")
print("\n0.5 = random   |   1.0 = perfect")

Cross-validated C-index (5-fold, mean +/- std):

  Expression only     : 0.582 +/- 0.048   (folds: ['0.604', '0.487', '0.617', '0.613', '0.588'])
  Mutation-ESM2 only  : 0.474 +/- 0.027   (folds: ['0.457', '0.442', '0.461', '0.493', '0.517'])
  Combined            : 0.587 +/- 0.042   (folds: ['0.596', '0.503', '0.616', '0.611', '0.607'])

0.5 = random   |   1.0 = perfect


## 9. Biological sanity check — TP53 mutation and survival

Before trusting the model, check the raw mutation signal is real: TP53 is the classic poor-prognosis driver in breast cancer. If TP53-mutant patients show worse survival here, the mutation features are capturing genuine biology.

Note: we flagged TP53 as mutant only when it has a damaging *missense* score, so nonsense/frameshift TP53 hits are not counted here — this undercounts true TP53 loss and is a conservative test.

In [22]:
tp53_mut = (X_mut_df.loc[common, "TP53"].values < 0)   # any damaging TP53 missense
kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(8, 5))
for grp, mask, color in [("TP53 mutant", tp53_mut, "#C0392B"),
                         ("TP53 wildtype", ~tp53_mut, "#2ECC71")]:
    kmf.fit(T[mask] / 30.44, E[mask], label=f"{grp} (n={int(mask.sum())})")
    kmf.plot_survival_function(ax=ax, ci_show=False, color=color)
lr = logrank_test(T[tp53_mut], T[~tp53_mut], E[tp53_mut], E[~tp53_mut])
ax.set_title(f"Survival by TP53 missense status  (log-rank p = {lr.p_value:.4f})", fontweight="bold")
ax.set_xlabel("Time (months)"); ax.set_ylabel("Survival probability"); ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.savefig("outputs/tp53_survival.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"TP53 mutant: {int(tp53_mut.sum())} | wildtype: {int((~tp53_mut).sum())} | log-rank p = {lr.p_value:.4f}")

KeyError: "['TCGA-A1-A0SB-01', 'TCGA-A1-A0SD-01', 'TCGA-A1-A0SG-01', 'TCGA-A1-A0SH-01', 'TCGA-A1-A0SJ-01', 'TCGA-A1-A0SM-01', 'TCGA-A1-A0SQ-01', 'TCGA-A2-A04P-01', 'TCGA-A2-A04Q-01', 'TCGA-A2-A04R-01', 'TCGA-A2-A04T-01', 'TCGA-A2-A04U-01', 'TCGA-A2-A04V-01', 'TCGA-A2-A04X-01', 'TCGA-A2-A04Y-01', 'TCGA-A2-A0CM-01', 'TCGA-A2-A0CP-01', 'TCGA-A2-A0CQ-01', 'TCGA-A2-A0CT-01', 'TCGA-A2-A0CU-01', 'TCGA-A2-A0CV-01', 'TCGA-A2-A0CX-01', 'TCGA-A2-A0D0-01', 'TCGA-A2-A0D1-01', 'TCGA-A2-A0D2-01', 'TCGA-A2-A0D4-01', 'TCGA-A2-A0EM-01', 'TCGA-A2-A0EO-01', 'TCGA-A2-A0EQ-01', 'TCGA-A2-A0ER-01', 'TCGA-A2-A0ES-01', 'TCGA-A2-A0ET-01', 'TCGA-A2-A0EU-01', 'TCGA-A2-A0EV-01', 'TCGA-A2-A0EX-01', 'TCGA-A2-A0EY-01', 'TCGA-A2-A0SU-01', 'TCGA-A2-A0SV-01', 'TCGA-A2-A0SW-01', 'TCGA-A2-A0T0-01', 'TCGA-A2-A0T2-01', 'TCGA-A2-A0T3-01', 'TCGA-A2-A0T5-01', 'TCGA-A2-A0YD-01', 'TCGA-A2-A0YE-01', 'TCGA-A2-A0YF-01', 'TCGA-A2-A0YG-01', 'TCGA-A2-A0YJ-01', 'TCGA-A2-A1FV-01', 'TCGA-A2-A1FW-01', 'TCGA-A2-A1FX-01', 'TCGA-A2-A1G4-01', 

## How to read this — and honest caveats

- **Combined > Expression, beyond the fold noise** -> the mutation-ESM2 features add independent prognostic signal. This is the result the reference-sequence notebook could not produce, because the features now differ between patients for reasons expression doesn't encode.
- **No improvement** -> still informative: with the small 8M ESM2 and missense-only scoring, the mutation signal may be too sparse (many patients carry no panel missense) to move overall survival, which is hard to shift in BRCA.

Caveats:

1. **Missense only.** The LLR score is defined for single-residue swaps, so nonsense/frameshift/indel drivers (including many TP53 and PTEN hits) are excluded. A fuller model would add a separate "truncating mutation" flag.
2. **Isoform mismatches.** UniProt canonical sequences don't always match the transcript the mutation was called on; those variants are skipped. Ensembl/RefSeq sequences matched to the MAF transcript would recover them.
3. **Model size.** `esm2_t6_8M` is the smallest ESM2. Variant-effect prediction improves with bigger models (`t33_650M`); worth trying if compute allows.
4. **Sparse features.** Only a handful of genes are mutated per patient, so the mutation matrix is mostly zeros — expected, but it limits what a linear Cox model can extract.

**Next steps:** add a truncating-mutation channel, widen the gene panel, or swap in a larger ESM2 and re-check whether the Combined advantage grows.